# 82514 · Sesión S18 — Bayes y Kalman: la puerta, el pasillo y la ganancia

**Bloque 6** · jueves 19 de noviembre de 2026 · 2 h  ·  IQS Universitat Ramon Llull

**Qué hace este cuaderno.** El robot móvil del almacén del curso razona con probabilidades: la regla de Bayes con el detector de la puerta estrecha del muro, dos medidas encadenadas (como en la parte D del examen final); el filtro de Bayes en el pasillo junto al muro, con un histograma de 130 celdas que pasa de plano a dos picos y de dos picos a uno; y Kalman en una dimensión por el mismo pasillo, con su ganancia. En casa, por qué el filtro complementario de S6 y el observador de S15 son Kalman.

**Se apoya en:** Thrun, Burgard y Fox (2005), caps. 2 a 4; Kalman (1960); documentación de numpy.

**Cómo usarlo en clase.** Sigue el guion de la sesión S18 en las diapositivas del bloque 6. Ejecuta la primera celda una sola vez al empezar; en Colab tarda unos segundos. Las celdas marcadas **Ejercicio** son para que los trabajen los estudiantes: las soluciones están al final del cuaderno.

---

In [ ]:
# Ejecutar una sola vez al abrir el cuaderno. En Colab tarda unos segundos: no instala nada.
import sys
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

VERSIONES = {'numpy': '2.4.6', 'matplotlib': '3.10.0'}   # las de todo el bloque 6 (las de Colab al prepararlo)
EN_COLAB = 'google.colab' in sys.modules
np.set_printoptions(precision=3, suppress=True)
plt.rcParams['figure.figsize'] = (10, 3.4)
plt.rcParams['axes.grid'] = True
IQS_AZUL, IQS_VERDE, GRIS, AMBAR, ROJO = '#1B2A80', '#1FA355', '#8A8FA8', '#B9770E', '#C0392B'
COMA = FuncFormatter(lambda v, p: f'{v:g}'.replace('.', ',').replace('-', '−'))   # ejes con coma decimal

def coma(x, n=2):
    """Un número con coma decimal y signo menos, para imprimir."""
    return f'{x:.{n}f}'.replace('.', ',').replace('-', '−')

def cien(v, n=2):
    """Un número pequeño o grande con coma decimal y potencia de diez: 1,40·10⁻¹⁰."""
    m, e = f'{v:.{n}e}'.split('e')
    return m.replace('.', ',').replace('-', '−') + '·10' + str(int(e)).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))

def ejes(*axs):
    for ax in axs:
        ax.xaxis.set_major_formatter(COMA); ax.yaxis.set_major_formatter(COMA)

print('numpy', np.__version__, '· matplotlib', matplotlib.__version__, '· en Colab:', EN_COLAB)
if not np.__version__.startswith('2.4'):
    print('AVISO: este cuaderno se probó con numpy', VERSIONES['numpy'], '; las cifras con semilla pueden cambiar un poco.')

> **Aviso para Colab.** Ábrelo al empezar la sesión y ejecuta la primera celda: no instala nada (usa numpy y matplotlib de Colab) y tarda unos segundos. Se probó con numpy 2.4.6 y matplotlib 3.10.0, las de todo el bloque 6; si tu numpy es otra, la celda avisa.
>
> **En clase (seis minutos):** las secciones 1 a 3, cambiando un número en cada una (la teoría de cada sección es la de las diapositivas 47 a 52: en clase, ejecuta sus celdas y ve directo a los párrafos **En clase**); se ponen en común a la vuelta del descanso. Al acabar, abre el cuaderno S18b y ejecuta su primera celda.
>
> **En casa:** la sección 4 y los ejercicios 1 a 3; las soluciones están al final.
>
> **No se sube:** este cuaderno no tiene tarea en Moodle.

## 1. ¿Está abierta la puerta estrecha?

*Diapositivas 47 y 48.*

El muro del almacén tiene dos puertas: la ancha, de 2 m, y la estrecha, de 0,8 m. La ruta del inventario baja por la estrecha, así que al robot le importa saber si está abierta. La mira con un detector que se equivoca: si la puerta está abierta, dice «abierta» con probabilidad **0,6** (y «cerrada» con 0,4); si está cerrada, dice «abierta» con **0,2**, la falsa alarma (y «cerrada» con 0,8). Son los números de Thrun et al. (2005).

- **Una distribución:** un número de 0 a 1 por estado, y suman 1. Sin mirar, p(abierta) = p(cerrada) = 0,5.
- **Una condicionada, p(z | x):** la probabilidad de medir z si el estado es x. La del detector es esta tabla (cada columna suma 1):

  | p(z \| x) | La puerta está abierta | La puerta está cerrada |
  |---|---|---|
  | El detector dice «abierta» | 0,6 | 0,2 (falsa alarma) |
  | El detector dice «cerrada» | 0,4 | 0,8 |

- **La regla de Bayes** da la vuelta: de p(z | x), lo que se sabe del sensor, a p(x | z), lo que se cree del estado tras medir:

  p(x | z) = p(z | x) · p(x) / p(z) = η · p(z | x) · p(x), con p(z) = Σₓ p(z | x) · p(x).

  p(x) es el **a priori**; p(z | x), leída como función de x, la **verosimilitud**; p(x | z), el **a posteriori**. η = 1/p(z) es el **normalizador**: hace que todo sume 1.

**Encadenar.** La creencia de después de una medida es la de antes de la siguiente. Vale si **las medidas son independientes entre sí cuando se conoce el estado**: es la independencia condicional, la hipótesis de Markov de la medida, la que se usa al encadenar.

In [ ]:
def bayes_puerta(b, z, acierto=0.6, falsa=0.2):
    """Una medida del detector de la puerta estrecha, con la regla de Bayes.
    b: p(abierta) antes de la medida. z: lo que dice el detector, 'abierta' o 'cerrada'.
    acierto = p(«abierta» | abierta); falsa = p(«abierta» | cerrada), la falsa alarma.
    Devuelve p(abierta) despues de la medida y el normalizador η = 1/p(z)."""
    if z == 'abierta':
        p_si, p_no = acierto, falsa              # p(z | abierta) y p(z | cerrada)
    else:
        p_si, p_no = 1 - acierto, 1 - falsa
    eta = 1 / (p_si * b + p_no * (1 - b))       # 1 / p(z): la probabilidad total de la medida
    return eta * p_si * b, eta


def secuencia(medidas, acierto=0.6, falsa=0.2, b=0.5):
    """La creencia tras cada medida, encadenadas: la de despues de una es la de antes de la siguiente."""
    creencias = [b]
    for z in medidas:
        b, _ = bayes_puerta(b, z, acierto, falsa)
        creencias.append(b)
    return creencias

b1, eta = bayes_puerta(0.5, 'abierta')
print(f'p(«abierta») = 0,6·0,5 + 0,2·0,5 = 0,4  ->  η = {coma(eta)} ;  p(abierta | «abierta») = {coma(b1)}')
print('Tres «abierta» seguidas:   ', ' -> '.join(coma(v, 3) for v in secuencia(['abierta'] * 3)))
print('«abierta» y luego «cerrada»:', ' -> '.join(coma(v, 3) for v in secuencia(['abierta', 'cerrada'])))

**En clase.** ¿Cuántas «abierta» seguidas hacen falta para pasar de 0,99? Sube `N` hasta pasar, con el detector de la tabla (0,6 y 0,2) y con uno de 0,95 y 0,05.

In [ ]:
ACIERTO, FALSA = 0.6, 0.2          # p(«abierta» | abierta) y p(«abierta» | cerrada): probad también 0.95 y 0.05
N = 2                              # cuántas «abierta» seguidas: subidlo hasta pasar de 0,99

b = secuencia(['abierta'] * N, ACIERTO, FALSA)
print(f'Detector {coma(ACIERTO)} / {coma(FALSA)}, {N} «abierta»: ' + ' -> '.join(coma(v, 3) for v in b)
      + ('   (pasa de 0,99)' if b[-1] > 0.99 else '   (aún no pasa de 0,99)'))
fig, ax = plt.subplots(figsize=(1.6 + 0.9 * N, 2.6))
ax.bar(['sin mirar'] + [f'{i}.ª' for i in range(1, N + 1)], b, color=IQS_AZUL, width=0.55)
ax.axhline(0.99, color=GRIS, ls='--', lw=1)
for i, v in enumerate(b):
    ax.text(i, v + 0.03, coma(v, 3), ha='center', fontsize=9)
ax.set_ylim(0, 1.15); ax.set_ylabel('p(abierta)'); ax.yaxis.set_major_formatter(COMA)   # x: las medidas, sin formato
plt.show()

## 2. El filtro de Bayes en el pasillo junto al muro

*Diapositivas 49 y 50.*

El **filtro de Bayes** es la puerta con un robot que se mueve. La **creencia**, bel(x), es una distribución de probabilidad sobre el estado, condicionada a las medidas y a los controles pasados (la odometría). Cada paso tiene dos fases:

- **Predecir** con el modelo de movimiento: bel⁻(xₖ) = Σ p(xₖ | xₖ₋₁, δₖ) · bel(xₖ₋₁), la probabilidad total. Hace crecer la incertidumbre: ensancha.
- **Corregir** con la verosimilitud de la medida: bel(xₖ) = η · p(zₖ | xₖ) · bel⁻(xₖ). La reduce: estrecha. Y se normaliza.

Funciona por las **hipótesis de Markov**: el estado de ahora resume el pasado; la medida depende solo de él, y el movimiento, del estado anterior y del mando.

**El pasillo.** El de la nave de arriba, a 0,5 m del muro (y = 5,2 m), de x = 0 a 13 m, en **130 celdas de 10 cm**: la creencia es un número por celda, un **histograma**. El robot mira de lado con un ultrasonido, el de S6: frente a una puerta no vuelve eco y el detector dice «hueco» (con 0,8); frente al muro dice «pared», pero a veces el eco rebota en el muro liso como en un espejo y no vuelve: la falsa alarma, con 0,1. Avanza 0,5 m por paso: 5 celdas con 0,8, y 4 o 6 con 0,1 cada una.

In [ ]:
NC = 130                                         # el pasillo de y = 5,2 m: de x = 0 a 13 m, en celdas de 10 cm
XC = (np.arange(NC) + 0.5) * 0.1                 # el centro de cada celda
HUECO = ((XC >= 0.8) & (XC < 2.8)) | ((XC >= 9.2) & (XC < 10.0))   # frente a la puerta ancha o a la estrecha
ANCHA, ESTRECHA = (XC >= 0.8) & (XC < 2.8), (XC >= 9.2) & (XC < 10.0)


def corregir(bel, hueco, acierto=0.8, falsa=0.1):
    """Corregir con lo que dice el detector lateral («hueco» o «pared»): verosimilitud por la creencia, y normalizar."""
    p = np.where(HUECO, acierto, falsa)          # p(«hueco» | celda)
    ver = p if hueco else 1 - p
    bel = ver * bel
    return bel / bel.sum()


def predecir(bel, celdas=5, nucleo=(0.1, 0.8, 0.1)):
    """Predecir el avance de 0,5 m (5 celdas) con su ruido: 4, 5 o 6 celdas, con 0,1, 0,8 y 0,1.
    Es la suma de la probabilidad total: cada celda reparte su creencia entre las tres de destino."""
    nueva = np.zeros_like(bel)
    for desp, w in zip((celdas - 1, celdas, celdas + 1), nucleo):
        nueva[desp:] += w * bel[:NC - desp]      # lo que saldria del pasillo se pierde
    return nueva / nueva.sum()


def recorre(acierto=0.8, falsa=0.1, semilla=3, x0=0.45, pasos=20):
    """El robot recorre el pasillo desde x0, en pasos de 0,5 m, sin saber donde empieza (creencia plana).
    En cada sitio mira de lado y el detector dice «hueco» o «pared» (con sus errores). Devuelve, en cada paso,
    la x verdadera, la medida y la creencia ya corregida."""
    rng = np.random.default_rng(semilla)
    bel, x, hist = np.full(NC, 1 / NC), x0, []
    for k in range(pasos):
        hay = bool(HUECO[int(x / 0.1)])
        z = rng.random() < (acierto if hay else falsa)
        bel = corregir(bel, z, acierto, falsa)
        hist.append((x, z, bel.copy()))
        bel = predecir(bel)
        x += 0.5
    return hist


def masa_cerca(bel, x, d=0.3):
    """La creencia a ±d de la x verdadera."""
    return bel[np.abs(XC - x) <= d + 1e-9].sum()

print(f'{NC} celdas; frente a un hueco: {HUECO.sum()} ({ANCHA.sum()} la ancha y {ESTRECHA.sum()} la estrecha)')
bel = corregir(np.full(NC, 1 / NC), True)
print(f'Primera medida «hueco» desde la creencia plana: la ancha {coma(100*bel[ANCHA].sum(),1)} %, la estrecha {coma(100*bel[ESTRECHA].sum(),1)} %')

**En clase.** El robot recorre el pasillo sin saber dónde empieza. Desde x = 0,95 m, frente a la puerta ancha, es lo de la diapositiva 50. Sal ahora de x = 5,95 m, frente al muro (`X0 = 5.95`): ¿cuándo queda un solo pico, y por qué tarda más que desde la ancha? (En el ejercicio 2, con un detector peor.)

In [ ]:
DET = (0.8, 0.1)                   # p(«hueco» | hueco) y p(«hueco» | pared)
X0 = 0.95                          # de dónde sale el robot (m): probad 5.95, frente al muro
PASOS = int((13 - X0) / 0.5)       # hasta el final del pasillo, en pasos de 0,5 m

H = recorre(*DET, semilla=18, x0=X0, pasos=PASOS)
for x, z, b in H:
    print(f'x = {coma(x)} m  {"«hueco»" if z else "«pared»"}   a ±0,3 m: {coma(100*masa_cerca(b, x),1):>5} %'
          f'   ancha {coma(100*b[ANCHA].sum(),1):>5} %   estrecha {coma(100*b[ESTRECHA].sum(),1):>5} %')
# Cuatro momentos: el primero, el primer «hueco», cuando pasa del 70 % cerca de la verdad, y el último
primer_hueco = next((i for i, (x, z, b) in enumerate(H) if z), 0)
pasa_70 = next((i for i, (x, z, b) in enumerate(H) if masa_cerca(b, x) > 0.7), len(H) - 1)
paneles = sorted({0, primer_hueco, pasa_70, len(H) - 1})
fig, axs = plt.subplots(len(paneles), 1, figsize=(10, 1.3 * len(paneles)), sharex=True)
for ax, i in zip(axs, paneles):
    x, z, b = H[i]
    ax.axvspan(0.8, 2.8, color=IQS_VERDE, alpha=0.15); ax.axvspan(9.2, 10.0, color=IQS_VERDE, alpha=0.15)
    ax.bar(XC, b, width=0.1, color=IQS_AZUL); ax.axvline(x, color=ROJO)
    ax.set_yticks([]); ax.set_title(f'tras medir en x = {coma(x)} m', fontsize=10, loc='left')
axs[-1].set_xlabel('x a lo largo del pasillo (m); en verde, los huecos; en rojo, donde está de verdad'); ejes(axs[-1])
plt.tight_layout(); plt.show()

## 3. Kalman en una dimensión, por el mismo pasillo

*Diapositivas 51 y 52.*

Si la creencia tiene un solo pico, basta una **gaussiana**: dos números, la media μ y la varianza σ² (desde hoy, σ es la **desviación típica**; en S17 era la anchura de la campana del suavizado, el mismo número). Dentro de μ ± σ cae el 68 %, y dentro de μ ± 2σ, el 95 %. Si el sistema es lineal, el ruido gaussiano y la creencia inicial gaussiana, la creencia es gaussiana para siempre: es **Kalman**.

- **Predecir:** x̂⁻ = x̂ + δ y (σ⁻)² = σ² + q. Las varianzas se suman. El superíndice menos es lo predicho: (σ⁻)², la varianza predicha.
- **Corregir** con la medida z, de varianza r: la **ganancia** L = (σ⁻)²/((σ⁻)² + r); x̂ = x̂⁻ + L·(z − x̂⁻); σ² = (1 − L)·(σ⁻)².

Con un sensor bueno, L se acerca a 1 y el filtro se fía de la medida; con uno malo, a 0, y se fía de la odometría. El ejemplo de la diapositiva: el robot cree estar en 5,00 m con σ = 10 cm, y una baliza le dice 5,12 m con σ = 5 cm.

Después, el robot recorre el pasillo en 24 pasos de 0,5 m. La odometría de un paso tiene σ = 4,2 cm (q = 0,0018 m²: los 6 cm por metro de S6), y en cada paso mide su x con la baliza (r = 0,0025 m²).

In [ ]:
def kalman_1d(x, s2, delta, q, z, r):
    """Un paso de Kalman en una dimension. Predecir: x̂⁻ = x̂ + δ y (σ⁻)² = σ² + q (las varianzas se suman).
    Corregir con la medida z: L = (σ⁻)² / ((σ⁻)² + r); x̂ = x̂⁻ + L·(z − x̂⁻); σ² = (1 − L)·(σ⁻)²."""
    x_, s2_ = x + delta, s2 + q
    L = s2_ / (s2_ + r)
    return x_ + L * (z - x_), (1 - L) * s2_, L, s2_


def pasillo_kalman(q=0.0018, r=0.05**2, s0=0.10, pasos=24, semilla=25):
    """El robot recorre el pasillo en pasos de 0,5 m (la odometria, con σ = 4,2 cm por paso: q = 0,0018 m²) y en
    cada paso mide su x con σ = 5 cm (r = 0,0025 m²). Devuelve la verdad, la odometria sola, la medida sola
    y Kalman, con su σ y su L."""
    rng = np.random.default_rng(semilla)
    x = xo = xk = 0.5
    s2 = s0**2
    filas = []
    for k in range(pasos):
        d = 0.5 + rng.normal(0, np.sqrt(q))      # lo que dice la odometria; el robot avanza 0,5 m
        x += 0.5
        z = x + rng.normal(0, np.sqrt(r))        # la medida
        xo += d                                  # a estima: solo la odometria
        xk, s2, L, s2_ = kalman_1d(xk, s2, d, q, z, r)
        filas.append((x, xo, z, xk, np.sqrt(s2_), np.sqrt(s2), L))
    return np.array(filas)


def l_estacionaria(q, r, n=500):
    """La ganancia a la que llega Kalman cuando q y r no cambian: la de un filtro de reparto fijo."""
    s2 = 1.0
    for _ in range(n):
        s2_ = s2 + q
        L = s2_ / (s2_ + r)
        s2 = (1 - L) * s2_
    return L

x, s2, L, _ = kalman_1d(5.00, 0.10**2, 0.0, 0.0, 5.12, 0.05**2)
print(f'L = 0,01/(0,01 + 0,0025) = {coma(L)} ;  x̂ = {coma(x,3)} m ;  σ = {coma(np.sqrt(s2)*100)} cm  (menos que 10 y que 5)')

**En clase.** Cambia la baliza de 5 a 20 cm (`R_BALIZA = 0.20`) y vuelve a ejecutar: ¿a qué L llega, y cuánto se equivoca el filtro?

In [ ]:
R_BALIZA = 0.05                    # σ de la medida, en m: probad 0.20
Q_PASO = 0.0018                    # varianza de la odometría en un paso de 0,5 m

F = pasillo_kalman(Q_PASO, R_BALIZA**2, s0=0.10, pasos=24, semilla=25)
rmse = lambda e: np.sqrt((e**2).mean())
print('L paso a paso:', ', '.join(coma(v, 3) for v in F[:6, 6]), '...', coma(F[-1, 6], 3))
print(f'σ corregida al final: {coma(F[-1, 5]*100)} cm')
print(f'Error medio (RMSE) en los 24 pasos: a estima {coma(rmse(F[:,1]-F[:,0])*100,1)} cm ; la medida sola {coma(rmse(F[:,2]-F[:,0])*100,1)} cm ;'
      f' Kalman {coma(rmse(F[:,3]-F[:,0])*100,1)} cm')
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.2))
k = np.arange(1, len(F) + 1)
a1.plot(k, (F[:, 1] - F[:, 0]) * 100, color=ROJO, label='a estima')
a1.plot(k, (F[:, 2] - F[:, 0]) * 100, '.', color=AMBAR, label='la medida sola')
a1.plot(k, (F[:, 3] - F[:, 0]) * 100, color=IQS_AZUL, lw=2, label='Kalman')
a1.fill_between(k, -196 * F[:, 5], 196 * F[:, 5], color=IQS_AZUL, alpha=0.12, label='±1,96σ de Kalman')
a1.set_xlabel('paso'); a1.set_ylabel('error (cm)'); a1.legend(fontsize=8); ejes(a1)
a2.plot(k, F[:, 6], 'o-', color=IQS_VERDE); a2.set_ylim(0, 1); a2.set_xlabel('paso'); a2.set_title('la ganancia L'); ejes(a2)
plt.tight_layout(); plt.show()

## 4. Kalman, ya visto: el complementario de S6 y el observador de S15 (en casa)

*Diapositiva 55.*

Si q y r no cambian, la ganancia de Kalman llega a un valor fijo, la **L estacionaria**, que solo depende del cociente q/r: en régimen, L²/(1 − L) = q/r.

- **El filtro complementario de S6**: θ̂ = 0,98·(θ̂ + ω·Δt) + 0,02·θ_acel = θ̂⁻ + 0,02·(θ_acel − θ̂⁻). Es un Kalman con L fija en 0,02: la que sale con q/r = 4,08·10⁻⁴, es decir, si el acelerómetro es 49,5 veces más ruidoso, en σ, que un paso del giróscopo.
- **El observador de S15**: la copia del modelo, corregida por ℓ con lo que se separa de la medida, dx̂/dt = A·x̂ + B·V + ℓ·(q − C·x̂), con la tensión V y el ángulo medido q de S15. La misma forma que x̂ = x̂⁻ + L·(z − H·x̂⁻): ℓ salía de los polos (−20 y −25: ℓ = (40,4; 313)); Kalman la saca del ruido.
- La «prueba de Kalman» de S15 es la de la controlabilidad: el mismo Kalman (1960), otra cosa.

In [ ]:
for nombre, q, r in (('el 98/2 de S6', 4.08e-4, 1.0), ('baliza de 20 cm', 0.0018, 0.04), ('el pasillo', 0.0018, 0.0025),
                     ('baliza de 1 cm', 0.0018, 0.0001)):
    qr_ = cien(q / r) if q / r < 0.01 else coma(q / r, 3 if q / r < 1 else 1)
    print(f'{nombre:16s} q/r = {qr_:>9s}  ->  L estacionaria = {coma(l_estacionaria(q, r), 3)}')
qr = np.logspace(-5, 2, 200)
fig, ax = plt.subplots(figsize=(6, 3))
ax.semilogx(qr, [l_estacionaria(v, 1.0) for v in qr], color=IQS_AZUL)
ax.set_xlabel('q / r'); ax.set_ylabel('L estacionaria'); ax.yaxis.set_major_formatter(COMA); ax.grid(True)
plt.show()

## Ejercicios (en casa)

### Ejercicio 1 · Una puerta con dos medidas

Otro detector: p(«abierta» | abierta) = 0,8 y p(«abierta» | cerrada) = 0,4, con la creencia inicial en 0,5. El detector dice «abierta» dos veces. Calcula a mano, con η, la creencia tras cada medida, y compruébalo con `bayes_puerta`. ¿Qué hipótesis usas al encadenar las dos medidas? Describe una situación del almacén en la que no se cumpla.

In [ ]:
# Ejercicio 1
# b1, eta1 = bayes_puerta(0.5, 'abierta', 0.8, 0.4)
# b2, eta2 = bayes_puerta(b1, 'abierta', 0.8, 0.4)
# print(coma(b1, 3), coma(eta1, 3), coma(b2, 3), coma(eta2, 3))

### Ejercicio 2 · Un detector peor

Repite el recorrido del pasillo con el detector de 0,6 y 0,2. ¿Se localiza el robot? Cuenta en cuántas de 100 semillas acaba con más del 80 % de la creencia a ±0,3 m de la verdad, con cada detector, y explícalo con lo que multiplica cada medida.

In [ ]:
# Ejercicio 2
# for det in ((0.8, 0.1), (0.6, 0.2)):
#     ok = 0
#     for s in range(100):
#         h = recorre(*det, semilla=s, x0=0.95, pasos=19)
#         ok += masa_cerca(h[-1][2], h[-1][0]) > 0.8
#     print(f'detector {coma(det[0], 1)} / {coma(det[1], 1)}:', ok, 'de 100')

### Ejercicio 3 · El 98/2 de S6, como Kalman

En régimen, L²/(1 − L) = q/r. Comprueba con `l_estacionaria` que q/r = 4,08·10⁻⁴ da L = 0,02, y calcula cuántas veces más ruidoso, en σ, es el acelerómetro que un paso del giróscopo. ¿Qué reparto (α = 1 − L) habría usado S6 si el acelerómetro fuera solo 10 veces más ruidoso? ¿Y qué τ, a 100 Hz (τ = α·Δt/(1 − α))?

In [ ]:
# Ejercicio 3
# for veces in (49.5, 10):
#     qr = 1 / veces**2
#     L = l_estacionaria(qr, 1.0)
#     print(coma(veces, 1), 'veces:', 'L =', coma(L, 4), ' α =', coma(1 - L, 4), ' τ =', coma((1 - L) * 0.01 / L, 3), 's')

---

## Soluciones

**Ejercicio 1.** p(«abierta») = 0,8·0,5 + 0,4·0,5 = 0,6, η = 1/0,6 = 1,667, y p(abierta | «abierta») = 1,667·0,8·0,5 = **0,667**. La segunda: 0,8·0,667/(0,8·0,667 + 0,4·0,333) = 0,533/0,667 = **0,800** (η = 1,5). Al encadenar se usa la hipótesis de Markov de la medida: **las medidas son independientes entre sí si se conoce el estado**. No se cumple, por ejemplo, si alguien está parado en la puerta: el detector se equivoca dos veces seguidas por la misma causa, y sus errores ya no son independientes. Se arregla metiendo esa causa en el estado («hay alguien en la puerta»); no ajustando el ruido.

**Ejercicio 2.** Con 0,8 y 0,1 el robot acaba localizado en 68 de las 100 semillas; con 0,6 y 0,2, solo en 5. Cada «hueco» multiplica la creencia de las celdas de hueco frente a las de pared por 0,8/0,1 = 8 con el detector bueno, y solo por 0,6/0,2 = 3 con el malo; cada «pared», por 0,2/0,9 = 0,22 frente a 0,4/0,8 = 0,5. Con 3 y 0,5, las cuatro medidas de la puerta ancha y las dos de la estrecha no bastan para separar las hipótesis, y la creencia se queda repartida.

**Ejercicio 3.** q/r = 4,08·10⁻⁴ da L = 0,0200; σ_acel/σ_paso = 1/√(4,08·10⁻⁴) = 49,5. Con un acelerómetro solo 10 veces más ruidoso, q/r = 0,01, L = 0,0951 y α = 0,905: S6 habría repartido 90,5 y 9,5 %, con τ = 0,095 s en vez de 0,49 s. El 98/2 no es una costumbre: es lo que dicen los dos ruidos.

---

## Para llevarse de esta sesión

**Bayes**: la creencia se predice con el movimiento, que la ensancha, y se corrige con la medida, que la estrecha; las medidas se encadenan porque son independientes si se conoce el estado. Un detector de 0,5 y 0,5 no informa.

**Kalman**, en una dimensión: las varianzas se suman al predecir y la ganancia reparte al corregir, según los dos ruidos. Fusionar da menos error que cualquiera de los dos por separado. El complementario de S6 y el observador de S15 eran Kalman.

En el cuaderno S18b, lo mismo con matrices y con el robot entero: el EKF por la ruta del inventario.

*Cuaderno del curso 82514 Mecatrónica y Robótica · IQS Universitat Ramon Llull · curso 2026/27*

*© Guillermo Reyes Carmenaty · Publicado bajo [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es): puedes usarlo, adaptarlo y redistribuirlo, incluso con fines comerciales, siempre que reconozcas la autoría.*